In [ ]:
# ============================================================
# Grad-CAM vs Grad-CAM++ on MISCLASSIFIED images ( best Fold )

# ============================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

# -------------------- SETTINGS --------------------
FOLD         = 0
TARGET       = "pred"  
SORT_BY_CONF = True     
ROWS_PER_FIG = 6        
MAX_SHOW     = None     
n_cls = len(CLASSES)

# -------------------- LOAD FOLD-0 MODEL --------------------
model0 = build_model(pretrained=False)
ckpt = os.path.join(OUT_DIR, f"best_proposed_fold{FOLD}.pth")
model0.load_state_dict(torch.load(ckpt, map_location=device))
model0.eval()

# -------------------- CAM EXTRACTOR (BiFPN fused map) --------------------
class CamExtractor:
    def __init__(self, model):
        layer = model.fusion if model.use_bifpn else model.attns[-1]
        self.act = None
        self.h = layer.register_forward_hook(lambda m, i, o: setattr(self, "act", o))
    def remove(self):
        self.h.remove()

ext = CamExtractor(model0)

def _norm01(cam):
    mn = cam.flatten(1).min(1)[0].view(-1, 1, 1)
    mx = cam.flatten(1).max(1)[0].view(-1, 1, 1)
    return (cam - mn) / (mx - mn + 1e-8)

def compute_cams(model, x, target=None):
    x = x.to(device)
    with torch.enable_grad():
        logits = model(x)
        pred = logits.argmax(1)
        tgt = pred if target is None else target.to(device)
        score = logits.gather(1, tgt[:, None]).sum()
        A = ext.act
        g = torch.autograd.grad(score, A)[0]
    A = A.detach(); g = g.detach()
    probs = logits.detach().float().softmax(1)

    # Grad-CAM
    w = g.mean(dim=(2, 3), keepdim=True)
    cam_g = F.relu((w * A).sum(1))

    # Grad-CAM++
    g2, g3 = g ** 2, g ** 3
    sumA = A.sum(dim=(2, 3), keepdim=True)
    alpha = g2 / (2 * g2 + sumA * g3 + 1e-8)
    alpha = torch.where(g != 0, alpha, torch.zeros_like(alpha))
    w_pp = (alpha * F.relu(g)).sum(dim=(2, 3), keepdim=True)
    cam_pp = F.relu((w_pp * A).sum(1))

    size = x.shape[-2:]
    up = lambda c: _norm01(F.interpolate(c[:, None], size=size, mode="bilinear",
                                         align_corners=False)[:, 0])
    return up(cam_g), up(cam_pp), tgt, pred, probs

# -------------------- COLLECT MISCLASSIFIED --------------------
test_idx = np.array(folds[FOLD]["test"])
te_loader = make_loader(folds[FOLD]["test"], eval_tf, False)   # shuffle=False -> order = test_idx

errors = []
pos = 0
for xb, yb in tqdm(te_loader, desc=f"Fold {FOLD}: scanning test set"):
    cam_g, cam_pp, tgt, pred, probs = compute_cams(
        model0, xb, target=None if TARGET == "pred" else yb)
    for i in range(xb.shape[0]):
        t, p = int(yb[i]), int(pred[i])
        if p != t:
            errors.append({
                "dataset_idx": int(test_idx[pos + i]),
                "true": t, "pred": p,
                "true_name": CLASSES[t], "pred_name": CLASSES[p],
                "p_pred": float(probs[i, p]), "p_true": float(probs[i, t]),
                "img": xb[i].cpu(), "cam_g": cam_g[i].cpu(), "cam_pp": cam_pp[i].cpu(),
            })
    pos += xb.shape[0]
ext.remove()

n_test = len(test_idx)
print(f"\nFold {FOLD}: {len(errors)} misclassified / {n_test} test images "
      f"(error rate {100*len(errors)/n_test:.2f}%)")

if len(errors) == 0:
    print("No misclassified images in this fold.")
else:
    if SORT_BY_CONF:
        errors.sort(key=lambda e: -e["p_pred"])
    if MAX_SHOW is not None:
        errors = errors[:MAX_SHOW]

    # ---- summary CSV + confusion pairs ----
    summ = pd.DataFrame([{k: v for k, v in e.items() if k not in ("img", "cam_g", "cam_pp")}
                         for e in errors])
    summ.to_csv(os.path.join(OUT_DIR, f"misclassified_fold{FOLD}.csv"), index=False)
    print("\nMost frequent confusions (True -> Pred):")
    print(summ.groupby(["true_name", "pred_name"]).size()
              .sort_values(ascending=False).to_string())

    # ---- figures: Input | Grad-CAM | Grad-CAM++ ----
    def to_img(t):
        a = t.permute(1, 2, 0).numpy()
        return (a - a.min()) / (a.max() - a.min() + 1e-8)   # display-only rescale

    explained = "predicted (wrong)" if TARGET == "pred" else "true"
    n_figs = int(np.ceil(len(errors) / ROWS_PER_FIG))
    for fi in range(n_figs):
        chunk = errors[fi * ROWS_PER_FIG:(fi + 1) * ROWS_PER_FIG]
        fig, axes = plt.subplots(len(chunk), 3, figsize=(10, 3.3 * len(chunk)))
        axes = np.atleast_2d(axes)
        for r, e in enumerate(chunk):
            img = to_img(e["img"])
            axes[r, 0].imshow(img)
            axes[r, 1].imshow(img); axes[r, 1].imshow(e["cam_g"].numpy(), cmap="jet", alpha=0.5)
            axes[r, 2].imshow(img); axes[r, 2].imshow(e["cam_pp"].numpy(), cmap="jet", alpha=0.5)
            axes[r, 0].set_ylabel(f"True: {e['true_name']}\nPred: {e['pred_name']}\n"
                                  f"p(pred)={e['p_pred']:.2f}, p(true)={e['p_true']:.2f}",
                                  fontsize=9)
            if r == 0:
                axes[r, 0].set_title("Input"); axes[r, 1].set_title("Grad-CAM")
                axes[r, 2].set_title("Grad-CAM++")
            for a in axes[r]:
                a.set_xticks([]); a.set_yticks([])
        plt.suptitle(f"Fold {FOLD}: Misclassified Images (CAM w.r.t. {explained} class) "
                     f"[{fi+1}/{n_figs}]", fontsize=12, fontweight="bold")
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, f"misclassified_cam_fold{FOLD}_part{fi+1}.png"),
                    dpi=300, bbox_inches="tight")
        plt.show()

    print(f"\n✅ Saved in {OUT_DIR}: misclassified_fold{FOLD}.csv, "
          f"misclassified_cam_fold{FOLD}_part1..{n_figs}.png")